# 01 · Bronze: land the SAP extracts

Loads each SAP extract file (one per extraction CDS view, see `sap_cds/`) into a **bronze** Delta table.
Values stay exactly as SAP delivered them (all strings), so a malformed value can never fail the load;
typing and business rules happen in silver.

| Step | What happens |
|---|---|
| Read manifest | `manifest.json` gives the extraction date and the row count of every file |
| Load | `<TABLE>.csv` → `bronze_<table>` plus lineage columns `_load_id`, `_source_file`, `_extracted_as_of`, `_ingested_at` |
| Reconcile | loaded rows are compared with the manifest and logged to `bronze_load_log`; a mismatch fails the run |

In [ ]:
LANDING_PATH = "Files/landing/sap_p2p"  # Lakehouse Files folder that holds the extract
TABLE_FORMAT = "delta"

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

SAP_TABLES = [
    "LFA1", "MARA", "MAKT", "T001W", "T023T", "T024", "TCURR",   # master data and texts
    "EBAN", "EKKO", "EKPO", "EKET", "EKBE",                        # requisitions and purchase orders
    "RBKP", "RSEG", "BKPF", "BSAK",                                # invoices and payments
]

manifest = (spark.read.option("multiLine", True).json(f"{LANDING_PATH}/manifest.json")
            .first().asDict(recursive=True))
load_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Load {load_id} | {manifest['source_system']} company code {manifest['company_code']} "
      f"| extracted as of {manifest['as_of_date']}")

In [ ]:
results = []
for table in SAP_TABLES:
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("encoding", "UTF-8")
          .csv(f"{LANDING_PATH}/{table}.csv")
          .withColumn("_load_id", F.lit(load_id))
          .withColumn("_source_file", F.col("_metadata.file_name"))
          .withColumn("_extracted_as_of", F.to_date(F.lit(manifest["as_of_date"])))
          .withColumn("_ingested_at", F.current_timestamp()))
    name = f"bronze_{table.lower()}"
    df.write.mode("overwrite").format(TABLE_FORMAT).option("overwriteSchema", "true").saveAsTable(name)

    loaded = spark.table(name).count()
    expected = manifest["row_counts"][table]
    results.append((load_id, table, expected, loaded, loaded == expected))
    print(f"{name:<14} expected {expected:>8,}  loaded {loaded:>8,}  {'OK' if loaded == expected else 'MISMATCH'}")

In [ ]:
(spark.createDataFrame(results, "load_id string, table_name string, expected_rows long, loaded_rows long, row_count_ok boolean")
 .withColumn("logged_at", F.current_timestamp())
 .write.mode("append").format(TABLE_FORMAT).saveAsTable("bronze_load_log"))

failed = [table for _, table, _, _, ok in results if not ok]
if failed:
    raise ValueError(f"Row-count reconciliation failed for: {', '.join(failed)}")
print(f"All {len(results)} tables reconciled with the manifest.")